In [1]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

import os
from getpass import getpass
import pandas as pd
import json
import re
from tqdm import tqdm
from sklearn.metrics import classification_report
from json.decoder import JSONDecodeError

import nest_asyncio
from dotenv import load_dotenv

pd.set_option('max_colwidth', 400)

nest_asyncio.apply()

load_dotenv('.env', verbose=True)
from huggingface_hub import InferenceClient

Для работы с llm через API будем использовать **Hugging Face**

Создайте токен на https://huggingface.co/settings/tokens

**Способ 1 (безопасный):** Создайте файл .env и добавьте:
- `HF_TOKEN=ваш_токен`

**Способ 2:** Вставьте токен напрямую в ноутбук в следующей ячейке:
- `hf_token = "<ваш токен>"`

In [2]:
# Способ 1: Загрузка из переменных окружения 
hf_token = os.environ.get('HF_TOKEN')

# Способ 2: Вставить токен напрямую 
# hf_token = "<ваш токен>"

In [3]:
client = InferenceClient(token=hf_token)
model_name = "Qwen/Qwen2.5-7B-Instruct"

In [4]:
def clean_json_response(response: str) -> str:
    """
    Убираем в ответе llm markdown (```json и ```)
    """
    response = response.strip()
    if response.startswith('```json'):
        response = response[7:] 
    elif response.startswith('```'):
        response = response[3:] 
    
    if response.endswith('```'):
        response = response[:-3]
    
    return response.strip()

In [5]:
def get_completion(prompt: str, system_prompt=""):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": prompt
        },
    ]
    
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=1000,
        temperature=0.7,
        top_p=0.9,
    )
    return response.choices[0].message.content

In [7]:
get_completion("Привет!")

'Привет! Как я могу вам помочь сегодня?'

## Предсказание категории

In [8]:
categories = [
    "Ролики и скейтбординг",
    "Настольные игры",
    "Теннис бадминтон пинг-понг",
    "Пейнтбол и страйкбол",
    "Единоборства",
    "Бильярд и боулинг",
    "Фитнес и тренажёры",
    "Туризм",
    "Игры с мячом",
    "Зимние виды спорта",
    "Дайвинг и водный спорт",
    "Спортивное питание",
]

system_prompt = f"""Ты - ведущий менеджер маркетплейса (эксперт по разметке данных). Твоя задача - проанализировать текстовое название/описание товара от пользователя и отнести его к одной из категорий раздела "Спорт и отдых".

Вот исчерпывающий список доступных категорий (выбирай строго одну из них, копируй название символ в символ):
{", ".join(categories)}

ИНСТРУКЦИИ:
1. Внимательно прочитай название товара. Определи ключевое слово (предмет) и его назначение.
2. Даже если информации мало, сделай обоснованное предположение и выбери самую близкую категорию из списка.
3. Сначала кратко объясни свой выбор (напиши рассуждение), а затем выдай итоговую категорию.
4. Твой ответ должен быть СТРОГО валидным JSON без Markdown-разметки, приветствий и лишнего текста.

Формат ответа (JSON):
{{
  "reasoning": "твое краткое объяснение",
  "category": "Название категории из списка"
}}

ПРИМЕРЫ:
<text>Протеин сывороточный со вкусом шоколада 1кг</text>
{{"reasoning": "Товар является пищевой добавкой для набора мышечной массы спортсменами.", "category": "Спортивное питание"}}

<text>Палатка 4-местная кемпинговая водонепроницаемая</text>
{{"reasoning": "Палатка используется для походов, кемпинга и выживания на природе, что напрямую относится к туризму.", "category": "Туризм"}}

<text>Мяч волейбольный Mikasa MVA200</text>
{{"reasoning": "Волейбол — это командная игра, в которой главным снарядом является мяч.", "category": "Игры с мячом"}}

<text>Кимоно для дзюдо белое рост 140</text>
{{"reasoning": "Кимоно — это специализированная одежда для занятий восточными боевыми искусствами.", "category": "Единоборства"}}

<text>Гантели разборные 2 шт по 15 кг</text>
{{"reasoning": "Гантели используются для силовых тренировок и поддержания физической формы дома или в зале.", "category": "Фитнес и тренажёры"}}

Теперь классифицируй следующий товар:"""

In [9]:
# test_df и val_df лежат на степике
test_df = pd.read_csv('test_df_llm_generation_1.csv')[['title', 'item_id']]
val_df = pd.read_csv('val_df_llm_generation.csv')
test_df.sample(10)

,title,item_id
38,Чехол Remington,38
78,Кимоно Kudo wear б/у,78
67,Сумка и шары для боулинга,67
39,Лента для хоккейной клюшки,39
83,"Аварийный рацион питания, 48 упаковок",83
115,Бамбинтон СССР,115
28,Киевница,28
94,Защита голени,94
26,Поповский кий Баринова черный эбен,26
112,Мешок боксерский набивной напольный для дома и др,112


In [10]:
def process_dataframe_with_cache(test_df, system_prompt, cache_file='answer_cache.json'):
    key_to_answer = {}
    if os.path.exists(cache_file):
        with open(cache_file, 'r', encoding='utf-8') as f:
            key_to_answer = json.load(f)
    answers = []
    
    for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
        title = row['title']
        item_id = row['item_id']
        key = f"{system_prompt}_{title}"
        
        if key in key_to_answer:
            answer = key_to_answer[key]
        else:
            prompt = system_prompt + f"\n<text>{title}</text>"
            answer = get_completion(prompt)
            key_to_answer[key] = answer
            # Сохраняем кэш после каждого нового ответа
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(key_to_answer, f, ensure_ascii=False, indent=2)
        answers.append(answer)
    
    return answers

In [11]:
answers = process_dataframe_with_cache(test_df, system_prompt)
predicted_categories = []
for answer in answers:
    try:
        # Очищаем ответ markdown
        cleaned_answer = clean_json_response(answer)
        parsed_answer = json.loads(cleaned_answer)
        predicted_category = parsed_answer['category']
    except JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        print(f"Ответ: {answer[:100]}...")
        predicted_category = 'other'
    predicted_categories.append(predicted_category)

 24%|██▍       | 29/121 [00:40<02:08,  1.39s/it]


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-69b69725-7b03e92a089c2bd36f147950;9bc590c5-e178-4192-ad7b-0deecc2301b2)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [11]:
test_df_gpt = test_df.copy()
test_df_gpt['category'] = predicted_categories
test_df_gpt[['item_id', 'category']].to_csv('test_df_to_upload.csv', index=False)